In [24]:
import os
import re
import numpy as np
import pandas as pd
import datetime
import time
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)

In [25]:
# File Paths
current_dir = os.getcwd()
data_path = f"{current_dir}/raw data"
ma_path = f"{data_path}/MA"
ny_path = f"{data_path}/NY"
sector_paths = ["FI", "HSA", "IN", "PST", "RT"]

In [78]:
# Accessing Data from Files
markets = {ma_path: 'MA', ny_path: 'NY'}
sectors = os.listdir(ma_path)
date_regex = "[0-9]{4}-[0-9]{1,2}-[0-9]{1,2}"

master_dataset = pd.DataFrame()

for market in markets:
    for sector in sectors:
        occupations = os.listdir(f"{market}/{sector}")
        for occupation in occupations:
            occ_files = os.listdir(f"{market}/{sector}/{occupation}")
            for f in occ_files:
                date = re.search(date_regex, f)
                occ_data = pd.read_csv(f"{market}/{sector}/{occupation}/{f}")
                occ_data = occ_data.assign(sector = sector, occupation = occupation, market = markets[market], date_scraped = date.group())
                master_dataset = pd.concat([master_dataset, occ_data])

# write raw observations into csv
master_dataset.to_csv(f"{data_path}/master_dataset_raw.csv", index=False)
print(len(master_dataset))

146339


In [ ]:
# retrieve master dataset
master_dataset_filepath = f"{data_path}/master_dataset_raw.csv"
master_dataset_df = pd.read_csv(master_dataset_filepath)

# drop observations with missing values in key columns + outside of specified date range
master_dataset_df = master_dataset_df.dropna(subset=['date_posted', 'description', 'location', 'company'])
master_dataset_df = master_dataset_df[master_dataset_df['date_posted'] >= "2026-03-01"]

# drop unneccessary columns
cols_to_drop = ['company_logo', 'company_addresses', 'company_rating', 
    'company_reviews_count', 'currency', 'vacancy_count', 'experience_range',
    'company_url_direct', 'company_url', 'work_from_home_type', 'skills', 'emails',
    'company_description']
master_dataset_df = master_dataset_df.drop(cols_to_drop, axis=1, errors='ignore')

# drop duplicates
master_dataset_df = master_dataset_df.drop_duplicates(subset=['id'])

# write cleaned observations into csv
master_dataset_df.to_csv(f"{data_path}/master_dataset.csv", index=False)

print(len(master_dataset_df))

64796


In [ ]:
# DATA EXPLORATION

In [105]:
# display job postings occupation taxonomy by sector, labor market, site, and occupation
pd.set_option('display.max_rows', None)
result = master_dataset_df.groupby(
    ['sector', 'market', 'site', 'occupation'])['id'].count()
display(result)

sector  market  site       occupation
FI      MA      glassdoor  CSR-FI         196
                           FIA            144
                           FM             119
                           ISA             28
                           PFA            111
                           SCFS             4
                indeed     CSR-FI         752
                           FIA            452
                           FM             203
                           ISA             49
                           PFA            242
                           SCFS           154
                linkedin   CSR-FI         368
                           FIA            414
                           FM             249
                           ISA            190
                           PFA            137
                           SCFS            35
        NY      glassdoor  CSR-FI         245
                           FIA            342
                           FM             

In [107]:
# sector segmentation analysis
result = master_dataset_df.groupby('sector')['id'].count()
display(result)

# platform segmentation analysis
result = master_dataset_df.groupby('site')['id'].count()
display(result)

# market segmentation analysis
result = master_dataset_df.groupby('market')['id'].count()
display(result)

sector
FI     10314
HSA    17642
IN     15224
PST    14103
RT      7513
Name: id, dtype: int64

site
glassdoor    12626
indeed       32581
linkedin     19589
Name: id, dtype: int64

market
MA    27455
NY    37341
Name: id, dtype: int64

In [108]:
# occupation segmentation analysis
result = master_dataset_df.groupby('occupation')['id'].count()
display(result.sort_values(ascending=False))
print(min(result))

occupation
MSA        4426
PAD        3687
MGA        3582
GOM-PST    3545
RN         3217
CIS        3046
FIA        2890
CNA        2877
CSR-FI     2752
AAA        2690
GOM-RT     2554
MRS        2428
RIC        2238
MAS        2232
CAH        2204
SM         1990
SRS        1915
SWE        1844
MHC        1675
CSA        1649
FM         1631
STO        1452
LAW        1447
PFA        1425
PLA         995
PCA         977
RSP         913
ISA         829
SCFS        787
ED          509
CSR-RT      390
Name: id, dtype: int64

390


In [ ]:
# DATA CLEANING

In [334]:
# keyword banks by occupation
# FI
scfs_keywords = ['investment banker', 'investment banking', 'investment banking analyst', 'trader', 'broker', 'financial services sales agent']
fm_keywords = ['finance manager', 'financial manager', 'finance director', 'fp&a manager']
fia_keywords = ['financial analyst', 'investment analyst', 'credit analyst', 'risk analyst', 'portfolio analyst']
pfa_keywords = ['financial advisor', 'wealth advisor', 'client advisor', 'relationship manager', 'advisory', 'retirement']
csr_fi_keywords = ['client associate', 'client service associate', 'client service representative', 'account']

# PST
law_keywords = ['attorney', 'lawyer', 'legal counsel', 'litigation', 'associate']
swe_keywords = ['software engineer', 'developer', 'software engineering intern', "technical lead", "frontend", "backend", "devops", "automation engineer", "machine learning engineer"]
aaa_keywords = ['accountant', 'accounting', 'audit', 'auditor', 'assurance', 'bookkeeper', 'controller']
mga_keywords = ['business analyst', 'consultant', 'consulting', 'strategy']
gom_pst_keywords = ['vice president', 'director', 'head of', 'senior manager', 'product manager', 'project manager', 'operations manager']

# RT
rsp_keywords = ['retail sales associate', 'salesperson', 'retail sales', 'retail associate', 'merchandise associate', 'retail cosmetics']
cah_keywords = ['cashier']
sto_keywords = ['stocking', 'stocker']
gom_rt_keywords = ['shift supervisor', 'store manager', 'restaurant manager']
csr_rt_keywords = ['customer service representative', 'customer service associate', 'call center representative']

# HSA
cna_keywords = ['cna', 'certified nursing assistant', 'nursing assistant']
pca_keywords = ['pca', 'hha', 'home health aide', 'personal care assistant', 'personal care aide', 'caregiver']
rn_keywords = ['rn', 'registered nurse', 'nurse rn']
msa_keywords = ['medical secretary', 'medical receptionist', 'dental receptionist', 'patient service representative', 'patient care coordinator']
mas_keywords = ['medical assistant', 'clinical assistant', 'clinical practice assistant']

# IN
srs_keywords = ['sales representative', 'sales associate', 'sales rep']
pd_keywords = ['digital production', 'producer', 'creative director', 'creative producer']
mrs_keywords = ['marketing specialist', 'social media', 'marketing manager']
cis_keywords = ['computer systems', 'information systems', 'information technology', 'IT']
ed_keywords = ['edit', 'editor', 'editorial']


In [335]:
# FI filter by occupation keywords
scfs_df = master_dataset_df[master_dataset_df['occupation'] == 'SCFS']
pre_scfs_count = len(scfs_df)
scfs_df = scfs_df[scfs_df['title'].str.contains('|'.join(scfs_keywords), case=False, na=False)]

fm_df = master_dataset_df[master_dataset_df['occupation'] == 'FM']
pre_fm_count = len(fm_df)
fm_df = fm_df[fm_df['title'].str.contains('|'.join(fm_keywords), case=False, na=False)]

fia_df = master_dataset_df[master_dataset_df['occupation'] == 'FIA']
pre_fia_count = len(fia_df)
fia_df = fia_df[fia_df['title'].str.contains('|'.join(fia_keywords), case=False, na=False)]

pfa_df = master_dataset_df[master_dataset_df['occupation'] == 'PFA']
pre_pfa_count = len(pfa_df)
pfa_df = pfa_df[pfa_df['title'].str.contains('|'.join(pfa_keywords), case=False, na=False)]

csr_fi_df = master_dataset_df[master_dataset_df['occupation'] == 'CSR-FI']
pre_csr_fi_count = len(csr_fi_df)
csr_fi_df = csr_fi_df[csr_fi_df['title'].str.contains('|'.join(csr_fi_keywords), case=False, na=False)]

print(f"Num Observations Before Filtering: {pre_scfs_count + pre_fm_count + pre_fia_count + pre_pfa_count + pre_csr_fi_count}")
print(f"SCFS: {pre_scfs_count}, FM: {pre_fm_count}, FIA: {pre_fia_count}, PFA: {pre_pfa_count}, CSR-FI: {pre_csr_fi_count}")
print(f"Num Observations After Filtering: {len(scfs_df) + len(fm_df) + len(fia_df) + len(pfa_df) + len(csr_fi_df)}")
print(f"SCFS: {len(scfs_df)}, FM: {len(fm_df)}, FIA: {len(fia_df)}, PFA: {len(pfa_df)}, CSR-FI: {len(csr_fi_df)}")


Num Observations Before Filtering: 9485
SCFS: 787, FM: 1631, FIA: 2890, PFA: 1425, CSR-FI: 2752
Num Observations After Filtering: 1418
SCFS: 87, FM: 284, FIA: 519, PFA: 243, CSR-FI: 285


In [336]:
# FI site segmentation analysis by occupation (after keyword filtering)
scfs_result = scfs_df.groupby(['site'])['id'].count()
display("SCFS: ", scfs_result)

fm_result = fm_df.groupby(['site'])['id'].count()
display("FM: ", fm_result)

fia_result = fia_df.groupby(['site'])['id'].count()
display("FIA: ", fia_result)

pfa_result = pfa_df.groupby(['site'])['id'].count()
display("PFA: ", pfa_result)

csr_fi_result = csr_fi_df.groupby(['site'])['id'].count()
display("CSR-FI: ", csr_fi_result)

'SCFS: '

site
glassdoor     5
indeed       46
linkedin     36
Name: id, dtype: int64

'FM: '

site
glassdoor    100
indeed       121
linkedin      63
Name: id, dtype: int64

'FIA: '

site
glassdoor    153
indeed       206
linkedin     160
Name: id, dtype: int64

'PFA: '

site
glassdoor     29
indeed        60
linkedin     154
Name: id, dtype: int64

'CSR-FI: '

site
glassdoor     58
indeed       129
linkedin      98
Name: id, dtype: int64

In [337]:
# PST filter by occupation keywords
law_df = master_dataset_df[master_dataset_df['occupation'] == 'LAW']
pre_law_count = len(law_df)
law_df = law_df[law_df['title'].str.contains('|'.join(law_keywords), case=False, na=False)]

swe_df = master_dataset_df[master_dataset_df['occupation'] == 'SWE']
pre_swe_count = len(swe_df)
swe_df = swe_df[swe_df['title'].str.contains('|'.join(swe_keywords), case=False, na=False)]

aaa_df = master_dataset_df[master_dataset_df['occupation'] == 'AAA']
pre_aaa_count = len(aaa_df)
aaa_df = aaa_df[aaa_df['title'].str.contains('|'.join(aaa_keywords), case=False, na=False)]

mga_df = master_dataset_df[master_dataset_df['occupation'] == 'MGA']
pre_mga_count = len(mga_df)
mga_df = mga_df[mga_df['title'].str.contains('|'.join(mga_keywords), case=False, na=False)]

gom_df = master_dataset_df[master_dataset_df['occupation'] == 'GOM-PST']
pre_gom_count = len(gom_df)
gom_df = gom_df[gom_df['title'].str.contains('|'.join(gom_keywords), case=False, na=False)]

print(f"Num Observations Before Filtering: {pre_law_count + pre_swe_count + pre_aaa_count + pre_mga_count + pre_gom_count}")
print(f"LAW: {pre_law_count}, SWE: {pre_swe_count}, AAA: {pre_aaa_count}, MGA: {pre_mga_count}, GOM-PST: {pre_gom_count}")
print(f"Num Observations After Filtering: {len(law_df) + len(swe_df) + len(aaa_df) + len(mga_df) + len(gom_df)}")
print(f"LAW: {len(law_df)}, SWE: {len(swe_df)}, AAA: {len(aaa_df)}, MGA: {len(mga_df)}, GOM-PST: {len(gom_df)}")

Num Observations Before Filtering: 13108
LAW: 1447, SWE: 1844, AAA: 2690, MGA: 3582, GOM-PST: 3545
Num Observations After Filtering: 7622
LAW: 1080, SWE: 1271, AAA: 2195, MGA: 1726, GOM-PST: 1350


In [338]:
# PST site segmentation analysis by occupation (after keyword filtering)
law_result = law_df.groupby(['site'])['id'].count()
display("LAW: ", law_result)

swe_result = swe_df.groupby(['site'])['id'].count()
display("SWE: ", swe_result)

aaa_result = aaa_df.groupby(['site'])['id'].count()
display("AAA: ", aaa_result)

mga_result = mga_df.groupby(['site'])['id'].count()
display("MGA: ", mga_result)

gom_result = gom_df.groupby(['site'])['id'].count()
display("GOM-PST: ", gom_result)

'LAW: '

site
glassdoor    232
indeed       538
linkedin     310
Name: id, dtype: int64

'SWE: '

site
glassdoor    285
indeed       662
linkedin     324
Name: id, dtype: int64

'AAA: '

site
glassdoor    511
indeed       910
linkedin     774
Name: id, dtype: int64

'MGA: '

site
glassdoor    354
indeed       771
linkedin     601
Name: id, dtype: int64

'GOM-PST: '

site
glassdoor    198
indeed       492
linkedin     660
Name: id, dtype: int64

In [339]:
# RT filter by occupation keywords
rsp_df = master_dataset_df[master_dataset_df['occupation'] == 'RSP']
pre_rsp_count = len(rsp_df)
rsp_df = rsp_df[rsp_df['title'].str.contains('|'.join(rsp_keywords), case=False, na=False)]

cah_df = master_dataset_df[master_dataset_df['occupation'] == 'CAH']
pre_cah_count = len(cah_df)
cah_df = cah_df[cah_df['title'].str.contains('|'.join(cah_keywords), case=False, na=False)]

sto_df = master_dataset_df[master_dataset_df['occupation'] == 'STO']
pre_sto_count = len(sto_df)
sto_df = sto_df[sto_df['title'].str.contains('|'.join(sto_keywords), case=False, na=False)]

sup_df = master_dataset_df[master_dataset_df['occupation'] == 'GOM-RT']
pre_sup_count = len(sup_df)
sup_df = sup_df[sup_df['title'].str.contains('|'.join(gom_rt_keywords), case=False, na=False)]

csr_rt_df = master_dataset_df[master_dataset_df['occupation'] == 'CSR-RT']
pre_csr_rt_count = len(csr_rt_df)
csr_rt_df = csr_rt_df[csr_rt_df['title'].str.contains('|'.join(csr_rt_keywords), case=False, na=False)]

print(f"Num Observations Before Filtering: {pre_rsp_count + pre_cah_count + pre_sto_count + pre_sup_count + pre_csr_rt_count}")
print(f"RSP: {pre_rsp_count}, CAH: {pre_cah_count}, STO: {pre_sto_count}, GOM-RT: {pre_sup_count}, CSR-RT: {pre_csr_rt_count}")
print(f"Num Observations After Filtering: {len(rsp_df) + len(cah_df) + len(sto_df) + len(sup_df) + len(csr_rt_df)}")
print(f"RSP: {len(rsp_df)}, CAH: {len(cah_df)}, STO: {len(sto_df)}, GOM-RT: {len(sup_df)}, CSR-RT: {len(csr_rt_df)}")

Num Observations Before Filtering: 7513
RSP: 913, CAH: 2204, STO: 1452, GOM-RT: 2554, CSR-RT: 390
Num Observations After Filtering: 2791
RSP: 626, CAH: 1092, STO: 558, GOM-RT: 387, CSR-RT: 128


In [340]:
# RT site segmentation analysis by occupation (after keyword filtering)
rsp_result = rsp_df.groupby(['site'])['id'].count()
display("RSP: ", rsp_result)

cah_result = cah_df.groupby(['site'])['id'].count()
display("CAH: ", cah_result)

sto_result = sto_df.groupby(['site'])['id'].count()
display("STO: ", sto_result)

sup_result = sup_df.groupby(['site'])['id'].count()
display("GOM-RT: ", sup_result)

csr_rt_result = csr_rt_df.groupby(['site'])['id'].count()
display("CSR-RT: ", csr_rt_result)

'RSP: '

site
glassdoor     54
indeed       292
linkedin     280
Name: id, dtype: int64

'CAH: '

site
glassdoor    224
indeed       672
linkedin     196
Name: id, dtype: int64

'STO: '

site
glassdoor     87
indeed       340
linkedin     131
Name: id, dtype: int64

'GOM-RT: '

site
glassdoor     34
indeed       168
linkedin     185
Name: id, dtype: int64

'CSR-RT: '

site
glassdoor    36
indeed       46
linkedin     46
Name: id, dtype: int64

In [341]:
# HSA filter by occupation keywords
cna_df = master_dataset_df[master_dataset_df['occupation'] == 'CNA']
pre_cna_count = len(cna_df)
cna_df = cna_df[cna_df['title'].str.contains('|'.join(cna_keywords), case=False, na=False)]

pca_df = master_dataset_df[master_dataset_df['occupation'] == 'PCA']
pre_pca_count = len(pca_df)
pca_df = pca_df[pca_df['title'].str.contains('|'.join(pca_keywords), case=False, na=False)]

rn_df = master_dataset_df[master_dataset_df['occupation'] == 'RN']
pre_rn_count = len(rn_df)
rn_df = rn_df[rn_df['title'].str.contains('|'.join(rn_keywords), case=False, na=False)]

msa_df = master_dataset_df[master_dataset_df['occupation'] == 'MSA']
pre_msa_count = len(msa_df)
msa_df = msa_df[msa_df['title'].str.contains('|'.join(msa_keywords), case=False, na=False)]

mas_df = master_dataset_df[master_dataset_df['occupation'] == 'MAS']
pre_mas_count = len(mas_df)
mas_df = mas_df[mas_df['title'].str.contains('|'.join(mas_keywords), case=False, na=False)]

print(f"Num Observations Before Filtering: {pre_cna_count + pre_pca_count + pre_rn_count + pre_msa_count + pre_mas_count}")
print(f"CNA: {pre_cna_count}, PCA: {pre_pca_count}, RN: {pre_rn_count}, MSA: {pre_msa_count}, MAS: {pre_mas_count}")
print(f"Num Observations After Filtering: {len(cna_df) + len(pca_df) + len(rn_df) + len(msa_df) + len(mas_df)}")
print(f"CNA: {len(cna_df)}, PCA: {len(pca_df)}, RN: {len(rn_df)}, MSA: {len(msa_df)}, MAS: {len(mas_df)}")

Num Observations Before Filtering: 13729
CNA: 2877, PCA: 977, RN: 3217, MSA: 4426, MAS: 2232
Num Observations After Filtering: 6417
CNA: 1042, PCA: 741, RN: 2583, MSA: 513, MAS: 1538


In [342]:
# HSA site segmentation analysis by occupation (after keyword filtering)
cna_result = cna_df.groupby(['site'])['id'].count()
display("CNA: ", cna_result)

pca_result = pca_df.groupby(['site'])['id'].count()
display("PCA: ", pca_result)

rn_result = rn_df.groupby(['site'])['id'].count()
display("RN: ", rn_result)

msa_result = msa_df.groupby(['site'])['id'].count()
display("MSA: ", msa_result)

mas_result = mas_df.groupby(['site'])['id'].count()
display("MAS: ", mas_result)


'CNA: '

site
glassdoor    218
indeed       621
linkedin     203
Name: id, dtype: int64

'PCA: '

site
glassdoor    216
indeed       408
linkedin     117
Name: id, dtype: int64

'RN: '

site
glassdoor     394
indeed       1496
linkedin      693
Name: id, dtype: int64

'MSA: '

site
glassdoor    107
indeed       340
linkedin      66
Name: id, dtype: int64

'MAS: '

site
glassdoor    322
indeed       944
linkedin     272
Name: id, dtype: int64

In [343]:
# IN filter by occupation keywords
srs_df = master_dataset_df[master_dataset_df['occupation'] == 'SRS']
pre_srs_count = len(srs_df)
srs_df = srs_df[srs_df['title'].str.contains('|'.join(srs_keywords), case=False, na=False)]

pad_df = master_dataset_df[master_dataset_df['occupation'] == 'PAD']
pre_pad_count = len(pad_df)
pad_df = pad_df[pad_df['title'].str.contains('|'.join(pd_keywords), case=False, na=False)]

mrs_df = master_dataset_df[master_dataset_df['occupation'] == 'MRS']
pre_mrs_count = len(mrs_df)
mrs_df = mrs_df[mrs_df['title'].str.contains('|'.join(mrs_keywords), case=False, na=False)]

cis_df = master_dataset_df[master_dataset_df['occupation'] == 'CIS']
pre_cis_count = len(cis_df)
cis_df = cis_df[cis_df['title'].str.contains('|'.join(cis_keywords), case=False, na=False)]

ed_df = master_dataset_df[master_dataset_df['occupation'] == 'ED']
pre_ed_count = len(ed_df)
ed_df = ed_df[ed_df['title'].str.contains('|'.join(ed_keywords), case=False, na=False)]

print(f"Num Observations Before Filtering: {pre_srs_count + pre_pad_count + pre_mrs_count + pre_cis_count + pre_ed_count}")
print(f"SRS: {pre_srs_count}, PAD: {pre_pad_count}, MRS: {pre_mrs_count}, CIS: {pre_cis_count}, ED: {pre_ed_count}")
print(f"Num Observations After Filtering: {len(srs_df) + len(pad_df) + len(mrs_df) + len(cis_df) + len(ed_df)}")
print(f"SRS: {len(srs_df)}, PAD: {len(pad_df)}, MRS: {len(mrs_df)}, CIS: {len(cis_df)}, ED: {len(ed_df)}")

Num Observations Before Filtering: 11585
SRS: 1915, PAD: 3687, MRS: 2428, CIS: 3046, ED: 509
Num Observations After Filtering: 2967
SRS: 795, PAD: 381, MRS: 910, CIS: 540, ED: 341


In [344]:
# IN site segmentation analysis by occupation (after keyword filtering)
srs_result = srs_df.groupby(['site'])['id'].count()
display("SRS: ", srs_result)

pad_result = pad_df.groupby(['site'])['id'].count()
display("PAD: ", pad_result)

mrs_result = mrs_df.groupby(['site'])['id'].count()
display("MRS: ", mrs_result)

cis_result = cis_df.groupby(['site'])['id'].count()
display("CIS: ", cis_result)

ed_result = ed_df.groupby(['site'])['id'].count()
display("ED: ", ed_result)

'SRS: '

site
glassdoor    163
indeed       236
linkedin     396
Name: id, dtype: int64

'PAD: '

site
glassdoor     84
indeed       168
linkedin     129
Name: id, dtype: int64

'MRS: '

site
glassdoor    277
indeed       456
linkedin     177
Name: id, dtype: int64

'CIS: '

site
glassdoor    121
indeed       284
linkedin     135
Name: id, dtype: int64

'ED: '

site
glassdoor     94
indeed       145
linkedin     102
Name: id, dtype: int64

In [ ]:
# DATA SAMPLING

In [ ]:
# Data Structurs for Sampling
platforms = {
    "I": "indeed",
    "L": "linkedin",
    "G": "glassdoor"
}

sectors = {
    "FI": "Finance & Insurance",
    "PST": "Professional, Scientific, and Technical Services",
    "IN": "Information",
    "RT": "Retail Trade",
    "HSA": "Healthcare & Social Assistance"
}

occupations = {
    # Finance & Insurance (FI)
    "SCFS": {"sector_id": "FI", "role": "Securities, Commodities, and Financial Services", "keywords": scfs_keywords, "obs": 305},
    "FM": {"sector_id": "FI",  "role": "Financial Manager", "keywords": fm_keywords, "obs": 63},
    "FIA": {"sector_id": "FI",  "role": "Finance and Investment Analyst", "keywords": fia_keywords, "obs": 62},
    "PFA": {"sector_id": "FI",  "role": "Personal Financial Advisor", "keywords": pfa_keywords, "obs": 53},
    "CSR-FI": {"sector_id": "FI",  "role": "Customer Service Representative", "keywords": csr_fi_keywords, "obs": 55},

    # Professional Scientific Technical Services (PST)
    "LAW": {"sector_id": "PST", "role":  "Lawyers", "keywords": law_keywords, "obs": 82},
    "SWE": {"sector_id": "PST", "role":  "Software Developer", "keywords": swe_keywords, "obs": 72},
    "AAA": {"sector_id": "PST", "role":  "Accountants and Auditors", "keywords": aaa_keywords, "obs": 66},
    "MGA": {"sector_id": "PST", "role":  "Management Analyst", "keywords": mga_keywords, "obs": 61},
    "GOM-PST": {"sector_id": "PST", "role":  "General and Operations Manager", "keywords": gom_pst_keywords, "obs": 54},

    # Retail Trade (RT)
    "RSP": {"sector_id": "RT", "role": "Retail Salesperson", "keywords": rsp_keywords, "obs": 135},
    "CAH": {"sector_id": "RT", "role": "Cashier", "keywords": cah_keywords, "obs": 84},
    "STO": {"sector_id": "RT", "role": "Stockers and Order Fillers", "keywords": sto_keywords, "obs": 64},
    "GOM-RT": {"sector_id": "RT", "role": "First-Line Supervisors of Retail Sales Workers", "keywords": gom_rt_keywords, "obs": 31},
    "CSR-RT": {"sector_id": "RT", "role": "Customer Service Representative", "keywords": csr_rt_keywords, "obs": 16},

    # Healthcare and Social Assistance (HSA)
    "PCA": {"sector_id": "HSA", "role": "Home, Health, and Personal Care Aides", "keywords": pca_keywords, "obs": 184},
    "RN": {"sector_id": "HSA", "role": "Registered Nurse", "keywords": rn_keywords, "obs": 79},
    "CNA": {"sector_id": "HSA", "role": "Nursing Assistant", "keywords": cna_keywords, "obs": 37},
    "MSA": {"sector_id": "HSA", "role": "Medical Secretaries and Administrative Assistants", "keywords": msa_keywords, "obs": 18},
    "MAS": {"sector_id": "HSA", "role": "Medical Assistant", "keywords": mas_keywords, "obs": 17},

    # Information (IN)
    "SRS": {"sector_id": "IN", "role": "Sales Representatives of Services", "keywords": srs_keywords, "obs": 121},
    "PAD": {"sector_id": "IN", "role": "Producers and Directors", "keywords": pd_keywords, "obs": 64},
    "MRS": {"sector_id": "IN", "role": "Market Research Analysts and Marketing Specialists", "keywords": mrs_keywords, "obs": 56},
    "CIS": {"sector_id": "IN", "role": "Computer and Information Systems Manager", "keywords": cis_keywords, "obs": 51},
    "ED": {"sector_id": "IN", "role": "Editor", "keywords": ed_keywords, "obs": 43},
}

In [ ]:
# Sample from filtered dataset for link checking
for occ in occupations:
    # filter master dataset by occupation and keywords
    sample_df = master_dataset_df[master_dataset_df['occupation'] == occ].reset_index(drop=True)
    sample_df = sample_df[sample_df['date_posted'].notna()]
    sample_df = sample_df[sample_df['title'].str.contains('|'.join(occupations[occ]['keywords']), case=False, na=False)]
    result = sample_df.groupby(['site'])['id'].count()

    # create sample dataset for link checking with additional columns for tracking listing status and age
    sample_df = sample_df[['id', 'site', 'job_url', 'date_posted']]
    sample_df= sample_df.assign(
        status="", 
        last_checked_date="", 
        listing_age=0, 
        listing_age_days=0
    )

    # extract sector and number of observations for occupation from data structure
    sector = occupations[occ]['sector_id']
    num_obs = occupations[occ]['obs']

    # for each platform, write filtered dataset to csv and randomly select job listings for link checking, writing sample to separate csv
    for p in platforms:
        if platforms[p] == "linkedin":
            sample_df = sample_df.assign(
                reposted=0,
                reposted_days=0
            )
   
        platform_sample_df = sample_df[sample_df['site'] == platforms[p]].reset_index(drop=True)
        filepath = f"{current_dir}/filtered data/{sector}/{sector}_{p}_{occ}.csv"
        platform_sample_df.to_csv(filepath, index=False)

        # randomly select job listings for link checking 
        df = pd.read_csv(filepath)
        samplepath = f"{current_dir}/sample data/{sector}/{sector}_{p}_{occ}_sample.csv"
        if len(df) >= num_obs:
            df = df.sample(n=num_obs).reset_index(drop=True)
        else:
            print(f"{occupations[occ]['role']} {platforms[p]} sample size: {len(df)}")
        df.to_csv(samplepath, index=False)

SCFS:
site
glassdoor     5
indeed       46
linkedin     36
Name: id, dtype: int64

Securities, Commodities, and Financial Services indeed sample size: 46
Securities, Commodities, and Financial Services linkedin sample size: 36
Securities, Commodities, and Financial Services glassdoor sample size: 5
FM:
site
glassdoor    100
indeed       121
linkedin      63
Name: id, dtype: int64

FIA:
site
glassdoor    153
indeed       206
linkedin     160
Name: id, dtype: int64

PFA:
site
glassdoor     29
indeed        60
linkedin     154
Name: id, dtype: int64

Personal Financial Advisor glassdoor sample size: 29
CSR-FI:
site
glassdoor     58
indeed       129
linkedin      98
Name: id, dtype: int64

LAW:
site
glassdoor    232
indeed       538
linkedin     310
Name: id, dtype: int64

SWE:
site
glassdoor    285
indeed       662
linkedin     324
Name: id, dtype: int64

AAA:
site
glassdoor    511
indeed       910
linkedin     774
Name: id, dtype: int64

MGA:
site
glassdoor    354
indeed       771
linked

In [ ]:
sample_df = sample_df[['id', 'site', 'job_url', 'date_posted']]
sample_df= sample_df.assign(
    status="", 
    last_checked_date="", 
    listing_age=0, 
    listing_age_days=0
)
sample_df

,id,site,job_url,date_posted,status,last_checked_date,last_active_date,listing_age,listing_age_days
0,gd-1010053177272,glassdoor,https://www.glassdoor.com/job-listing/j?jl=1010053177272,2026-03-04,,,,0,0
1,gd-1010061168451,glassdoor,https://www.glassdoor.com/job-listing/j?jl=1010061168451,2026-03-11,,,,0,0
2,gd-1010060927691,glassdoor,https://www.glassdoor.com/job-listing/j?jl=1010060927691,2026-03-11,,,,0,0
3,gd-1010060523034,glassdoor,https://www.glassdoor.com/job-listing/j?jl=1010060523034,2026-03-11,,,,0,0
4,in-42232e697d9a6a7e,indeed,https://www.indeed.com/viewjob?jk=42232e697d9a6a7e,2026-03-05,,,,0,0
...,...,...,...,...,...,...,...,...,...
1773,li-3672495974,linkedin,https://www.linkedin.com/jobs/view/3672495974,2026-03-23,,,,0,0
1774,li-4389075815,linkedin,https://www.linkedin.com/jobs/view/4389075815,2026-03-23,,,,0,0
1775,li-4207821534,linkedin,https://www.linkedin.com/jobs/view/4207821534,2026-03-23,,,,0,0
1776,li-4389091995,linkedin,https://www.linkedin.com/jobs/view/4389091995,2026-03-23,,,,0,0


In [13]:
today = datetime.datetime.now().strftime("%Y-%m-%d")
today_date_obj = datetime.datetime.strptime(today, "%Y-%m-%d")

In [118]:
# create csv file for editor occupation to be used for link checking
filepath = f"{current_dir}/filtered data/FI_I_SCFS.csv"
sample_df = sample_df[sample_df['site'] == 'indeed'].reset_index(drop=True)
sample_df.to_csv(filepath, index=False)

In [ ]:
# randomly select job listings for link checking - write into a function
filepath = f"{current_dir}/filtered data/FI_L_SCFS.csv"
df = pd.read_csv(filepath)
samplepath = f"{current_dir}/sample data/FI_L_SCFS_sample.csv"
df = df.sample(n=304).reset_index(drop=True)
df.to_csv(samplepath, index=False)

In [114]:
# editor link checking breakdown
filepath = f"{current_dir}/filtered data/FI_G_FIA.csv"
links_df = pd.read_csv(filepath)

# for i in range(len(links_df)):
#     platform = links_df.at[i, 'site']
#     url = links_df.at[i, 'job_url']

#     date_posted = datetime.datetime.strptime(links_df.at[i, 'date_posted'], "%Y-%m-%d")
#     listing_age = today_date_obj - date_posted
#     days_old = str(listing_age).split(" ")[0]
#     links_df.at[i, 'listing_age_days'] = int(days_old)

#     weeks_old = int(days_old) // 7
#     links_df.at[i, 'listing_age'] = weeks_old + 1
#     links_df.to_csv(filepath, index=False)

status_result = links_df.groupby(['status'])['id'].count()
display(status_result)
listing_age_result = links_df.groupby(['listing_age'])['id'].count()
display(listing_age_result)
listing_age_day_result = links_df.groupby(['listing_age_days'])['id'].count()
display(listing_age_day_result)

status
active     57
expired    43
Name: id, dtype: int64

listing_age
0    504
2     43
3     42
Name: id, dtype: int64

listing_age_days
0     489
4      10
5       5
14      3
15     10
16      4
17     10
18     10
19      6
22      4
23     24
24      2
25      3
26      9
Name: id, dtype: int64

In [60]:
links_active_df = links_df[links_df['status'] == 'active']
links_active_df

,Unnamed: 0,id,site,job_url,job_url_direct,date_posted,status,last_checked_date,last_active_date,listing_age,listing_age_days
1,1,gd-1010056428185,glassdoor,https://www.glassdoor.com/job-listing/j?jl=1010056428185,NaN,2026-03-06,active,2026-03-29,NaN,3,23
2,2,gd-1010056181720,glassdoor,https://www.glassdoor.com/job-listing/j?jl=1010056181720,NaN,2026-03-06,active,2026-03-29,NaN,3,23
3,3,gd-1010056142664,glassdoor,https://www.glassdoor.com/job-listing/j?jl=1010056142664,NaN,2026-03-06,active,2026-03-29,NaN,3,23
4,4,gd-1010054369204,glassdoor,https://www.glassdoor.com/job-listing/j?jl=1010054369204,NaN,2026-03-05,active,2026-03-29,NaN,3,24
6,6,gd-1010054353079,glassdoor,https://www.glassdoor.com/job-listing/j?jl=1010054353079,NaN,2026-03-05,active,2026-03-29,NaN,3,24
...,...,...,...,...,...,...,...,...,...,...,...
147,147,gd-1010074618240,glassdoor,https://www.glassdoor.com/job-listing/j?jl=1010074618240,NaN,2026-03-24,active,2026-03-29,NaN,0,5
148,148,gd-1010074446854,glassdoor,https://www.glassdoor.com/job-listing/j?jl=1010074446854,NaN,2026-03-24,active,2026-03-29,NaN,0,5
149,149,gd-1010074838964,glassdoor,https://www.glassdoor.com/job-listing/j?jl=1010074838964,NaN,2026-03-24,active,2026-03-29,NaN,0,5
150,150,gd-1010074534093,glassdoor,https://www.glassdoor.com/job-listing/j?jl=1010074534093,NaN,2026-03-24,active,2026-03-29,NaN,0,5
